# Drum: konteks maksimal 90 detik lalu stitching

Eksperimen pembanding setelah drum full-song satu konteks ditolak. Pakai **VM Colab aktif yang memiliki runtime/bobot ACE-Step**; salin sel ini ke notebook aktif agar cache tetap dipakai. Helper tersemat; tidak perlu ZIP tools. Piano asli dipakai kembali jika tersedia.

Setting model tetap sama: acoustic drum caption, Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah. Batas **90 detik termasuk margin enam detik dan padding model**, bukan core 90 detik ditambah margin. Untuk piano 4:03: tiga konteks model 90/89/90 detik dan dua sambungan. Crossfade float equal-power .58 detik; kualitas belum terbukti. Groove/timbre antar-chunk masih dapat berubah.

Tidak mengulang bass atau mengganti baseline 30 detik. DRY_RUN=True hanya membuat plan. Notebook ini tidak memasang atau mengunduh ulang model. VM kosong perlu sel setup/runtime/bobot notebook 12 terlebih dahulu, tanpa menjalankan generasi bassnya.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid, zipfile
from google.colab import files
DRY_RUN = False
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "electric_full_song.py": {
    "sha256": "04d99be5e6bf1635b7ac16347680f344be13344fa1b8953523c9543a96c502ce",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwXVyr58lJikRTMGpurAjc61gtkC3/GWE65CK9ojEGYB5lkohzY6v379aqRDJ5jxyVVkpV/4bJiI4FwjJy6wyViLFFjktsrBqu7AHgk4+FVaMIIOUKVoefIkjB25dg6R4lX19z6F+S8EUXs/Ut7ktLYsO95MK0478LfP22Q3YZ3B8jJY9XudOqgJXsFG5WlCpzDOZ7eL5tlDp32j7JNZKSTXeXfuv/v8i396cvqOVB58/+vZp/r7w/X7X05O/3Z2+VZ/nV5dnl/8+vbio2PZ72nh6c1ucWmMdFqbc9tBc6C3zjBE7+oDXJ2fv7+4PJs6x7DPp48nlzfnVx8/nH28sXu0vmvA0N0T1cYjB5hwaYRiPcCuzEvU+na4tW++1rAm857m4dI2EanBw+QOBrorN+K5aPfp57cnYp3Ubgcm8yjyrTFHyu/0+vPow/WNmAdxPAvCe6dxFp0x8cCKi2KgWIz0Aj/rT3skzoWrFXYb7IMzmKsPtK+Bme1zTsK/42Vca0nvJJQIF91lXRElkk+Z9RVIO58FLVKyX6d43ELGc5BNQb4A4Xt4eP+Av7oZLm5nDtb1AsS9XDIw6QWnXYGiRYn2hAqBoHaa8Qk8iRdh7QRQKWNQlx40hmU5Htu5vTY1CnfC3uit+jRCP3v028nZtC5dmMfrYml5Ga3YdrHOKM/6LLjagGoCG8DjbwLp0y00aBlaWpSb/brX241RgEYT+824iCYjGGUICuf/wr8tCjKVP7rHbS/QJhwfgwq+/yICqcBnJu0EU+PobOZTAhlQz18+gq4xnAKL7BDsPwX+4Y7bqm4EJR7TntXWmQUE3lK9/an6vp1aE2lHEl/N9m4sTpNCCxr7QdcGFN6OvnQFRYcAYEAL8TTyL+IUpBf6JdpkccEj26QKfERTOvWdXbDFfgDVax2xqca5FU9rORRXnp0Hgn07aSFjZpD93nA2TSNbIgud00+bIZa6uT1Mx236IjVg7shcaf2rz5nFQeKaLbUmss2xIVYytacbtYjpjd/TmTFYMcFheNyvbJCpqaRCriZBMjVEyNVYOzww4I1GLZQOQTXLqaaYMINVpk5Bcs9Hued0XCYLigA86yLanq1btKUExAYwapWruUadT/erc016Uw3MWmKysVvddjdABNfygPBN1BygWG24p9ZHy1YN7YMWEDX0gIKmNTCGFQVOaxLoQ2q1Whv6WMqGomSdg6CnOE4+tXw67tH4KPNtLT00+RmOYz5vowEjQlmJ5ZnyjH4iAtEkQJ1qrdUhTHCAynUxdeZgvAAnDXnGdO48kjKiL5TQaPP5/hN4stTUKPJBJc62KEqk2Dr7NxKWcYvQuGE52MkNh+kaWY6Swcd2SuhFSZ7/jxxilQ8MQIyupuigsm88faTDPw1bicL94VJr3T+cHFTzdloQbDA6xD7n6YqpMwI/nEpAkPPyjdSFvEFYVkqIV+cVHYM+kAP3wUJqx/gPowj9mibSycNxq0pVP8bKxoSiLsO6gNWHc26UfPBW2WtEoeus1BcqBTEj8bsxyDII/g8Io2LyvVnmBmZjNYODxQBTWmnGrcf/9Oq+veZunIOUAriibOZfVLhTuE2nUReEaMhW9TKgkop1XHpfVeYgBCvYopPeAXgTDw1oD/bc1t5vLmWEuqC7o42oXtxZyKMghq6E9v5NZedYqlSBiu/rfKGgIQeBMctYjb+49t+enb8HxnvLlTZga6tNy4ybU21qDHJswx73oYYz2rpksE3MRA751vubacPql+GUJoImdqTItbV1VatqVNCoEWqyTJ1RR742bmGPLBgOfYuMx2P/hHDuY7H2eKTJwgzVSmuk/9axSFPdRAqlwmxTDDiO8zcpMxGIYgUaQ2DUGXUvuc8MDhHMMa9xdfVhhEvB+vWzgFVQUCE/VumEwRo0exWOaickdtBYo7YNgwhkBRbuc9SEY/8UIjIhmz7ctys4GyRsFU7uJDUmdRqtsaGxKnVwsV1pcbir2GJoLH1tHWvi7FkEs0ZB5oLQmmGl30RXJltFyMDXfRMHfyzkdnr14cPFp5vh3kqPb4i5sWYio9XXKsCaAp7IYiGjxg5VEBAL0iO0BG2q0jkBC1msYZGMLtPyHHRlZAprdOoZp82xHawsnm7MLLZ6aSWr8NOYuI23G8yFVhG5mWiVnWMq0qp7sAdhxsvHSIiuC+Sw7j4zofE2yERsIjWfY9U9P2wBg6GgQ4IDZxIutSLSOjbiCOafXaM/ade+mwih2XbQxZF9PSp+Nynfxhsm/Z5BPz/iWHqab7+tNkALPcr3kUmOeSWW87CXeZBDySbLfWNdgHHrCT1W8iKQqvjDZbx11K9jHDMfM+kK52mGuiUfDDN+LY+qk/B51sPq7spRMXyOVdTurs65YQUv3gufcmGlpkrW6ZoicvWTLkoeRjIDIxpZuX7OhdRgXnKJ307OqHCkVBSEBfVB7ogn3gV59AAoAAelJOI372dgg4Ksl3CLT8JAEyUS8/xcIldKQrTn7En29KhnRmWPJ2el5/pL3hpBjaFoFY7apbW4Q9uFwwBFQhPJLJg2U0/fegOLc7QqmPQEVufO9fuTywm9vXs0G99+10zbfXf3VBicUkz81+vPonrd9Xy4lY910N756vJMXJ/c3Ezsnes9J97r+ZMunB4+czgdtmcKrV8OVjS244xW0Q5a593KfV0KTirRJ2Hj+5ZmJIfiiNN/ZPEf4SynTmjwAMBZw/4f1CFJIhUKj7Hgk/ncJ3cMA6w13VSKrC6kAT043Zlh6roVNfndOnxgE/6gEGbP7qYCTWvPTnjNUkRuX9iuY8sOGkrKJlJrJKUctCTllIMxgSIHc4v4gqQefuuA9i6wXA5GHr+4/j0lkYU+PVXDW/kEJiSVbNKQY4/1OWEnDCp29r4d31nPmui5Un8cs2OXOxZm0ECnKKqxSapXUfB7SIHpBsj0DJOsJXVMRv6OnA2OgJVuW9UAdziiYq1253MAtTMx+kC8Mzs4aDOouWJIasfCAiiHsqkUBPQROby4BJxCliUVptzdOjwI9VtfL0+1QvzSoBgVcv2alLWqlRPQ71ixJmDY94B11/PV/koau76hgmWr9AG3LMH/oUA+6s1YLtLmK9sgbryG7WwDBjX4WT4/F1VfzZZkjeDqIOnQvbKawMUChi79GhLc07++GY13KHpWwQc462AhTdeTTZD4AIKRRSR4L7dEffR2gcxFaBnqTzAYDaI8he/63MGzj1dOWLc3HobQfsDDeaEN1x67le2skSm8ajARk4+WOfSsmc6MD7zw/ZrVO8PKA79QXzVTHbcHIIxyEMzpingbRumUEkHquct9TirCteKtuNCIdhYV0VvKi733nZK438vvl8Fk4aEaxB+ELfpBLxR5Nl2ShgHN3oLARfDBpwEP5QPg1vgsDTQoJjwxN2qxJnrEjdUaUl73sJBvDNN81JFo7ecwfZOI13hOIwfrmFJ3ulF7JSTlUAvSvSuEPRdhmwcvQmBkGMImg5c/z6pVE9bwsE4a6dyDaGYcDGCbdfJNod4uKLvrPCRqV6pVcSqrQG6wV1O8fMlG6XRr0UZh292zALsgJYPP5Bb8ugW9O1ubWznEga0i6vRE9b640i7opKKhhwK7Pn5fJLU+d6NM7m7HVvsX6+ltxGd3bVa/q3vuhhiD7WzWirE/f6V+F6a20YaVqJp2FTam8IiKfU3FU/33rmizpYN1rs62cjq9zs6wdSaDe37+DbyIth0YWbMtOIqNBXeOesnCOs2yf93moN3LEhkyI0w7vNSlHzD4EZO7JjRj+7v2ZHrDQkzrqSbMLHOQ7qbtzdCQF9PG9HmS+VbvtZWIOtjtVb4wi9pPu/+LDKoOcO+y99sOkC2x9dRGDKvpnvEneWeWgjfzetX4t0Iaox27Av+WVCEgVwDGMl2QseIv4iOQgOIXN1VmseLNEUt+tmikmez9wSz0i66j39/DWUxEnkpb0B8zZS7eSb4gW/uaetxIcgQS4+a+H6Uh+PfWTPwnHfxAT3Fr/x4pZHpNOQXYMgBGn+6KEeh//KK4Pb7bu3L1T15Ua+8dzjEF6xwmLGr5/Lt2ohpqPZnMmfoW470zrZdv9mT896KmztVKla1nfyZKtgq+qNV6ha92jAW7YwcdqqJaGkIKKOQU87hwqf0zqyhMz0R9xIoIbj5fo0d40yiF0evSX7iyybmib4PFkLyBXZPRqP6mMUgeduKEGgFV7X8RRDZ8ArNDUb1k48dc+kSSrR1DIthvkO20FTPco0oZVSca2svXhzK5I+5sxiy5sS6kN9kkatYfg1ZEz7khiTqpOPLWWMbo+/6H4LoE7DYnQJZVWHzIMpacB5/rMX30pYmTD/4HNKQt7d5MAAA="
  },
  "drum_probe.py": {
    "sha256": "e1197fddd057474b7cc0b9258bb0851a15996fd9e2d96281a99bd4fdf65dc50c",
    "gzip_base64": "H4sIAAAAAAAC/7Va3XPbNhJ/11+B5B5AthRlx9NmTj4+uI1z07lem0vSduY0Gg5MghJOFMkjQNlOxv/77eKD30rSds4PsQkuFov9/O0yz58/f1U3R1LV5R2XpKxJ1uT5UpbFjkhR7HK+rJiUpMEHovacVKIoeEpuvr9dvlO8It//8uqG1E2hxJGHz58/X4hjVdaKsHpXsVpy9/wf4LnI6hLOYmqfiztiX7yBR0ck940SefvU3IFcCZfSrXwQVSZyvnDPPOeJqkUSo9SxlppJK/hi8ebt7bvb9yQiHxcEfqhI6ZqypGykEskyhXvL5emSBvRYpjzX77iEO8HiN8s7Jjm8SvlJJBzeJU3KaGAYVTVPhBRlAetZXjJ19QJIRSEV8OSFgmXNHRYl56mk683F1u7dNSJlRcJjmbAcGL8ML3BrxmuuV+F8oP8GFssMeaexKuOkauj6fd1wyyRhlTLH/yDLnCkwiLsXwZPJQYAFkqQ8VqwQKFJIwEpwu5oJMF9ACqaamuVkV5fliZN7ofawKTkERBas5oQVKUnyUgLnvVjumQpIVuZ5ed/5AStKUvE6K+sjXoiAB8DLgFT7mml3QR7pY8GOIpEh+T7nrDbS7YWSAVEMLguGI2DSXGpqWYEJyB1X95wXhhGHreii4JxF/giCl+RUguq0sx55XqZw5071MgQjPS0Wi5Rn4GnJge24VzaqauAGsknQm/y11iIIxiNtqDgDK90BcQgORvVLrRDrbuG/RfUafls+K9wY0HsaoH5BqegJUUv7w5v41e3rH2/e377y0RtZnezFiZsz8QcUBqYAq8EqSE42RgYdgiGGCbgNsIXo4elKFHCiW93xgtcMLb/iD6B6Y1nzsmU/+elvk0baGYYXFxfxTazdNkanvVjBzZr8C9ijMzY1X9nfjrdhBRc5CX4fHqsrDDTxMFjZrgd8RUacjp2C/FDIGBXr+WuryfC+FoqPCYN2x8KystbujviUTY0N7JXRC/6Egb/UyFpD4T07Gc3ov0YawZ8vvbbxeWtS7kFgq0CWTZ3wwAbAV4FUrFbRRZA2xuzR1UXAH3jSKB69hqBCfi4VsSYVZfRTWXAbMFOO0Qazt1f5IWipzE9gJX3zCq88I4G/1YxMgg6ZUizZazJjNFMeakhIhc4obVVgL775NtD3j3UdaYlt4Yl5cRJ1WWA8uF0zr7p9kC440+ZAF+ltifNyt+NpR3lkSXxf1gfIxfd9OnANSHs9Qp1sax5D4ZCQs4tWfJusNRdeg4L/2wigy6qrF73dUF7itq64ra9//Pnm/dWL+Lef3/7j9u3g2h33ZM/T+D/lXVDlzKgmKYtM7CJU7IqaB7kaFj7jgy5UDE0XajoZI0WIFUh69j3oDMoRf1Ce7z+LTHnt/BVKi+TkV5Y3/Lauy9qjFlpwyRVJRQaOJa/Ncw1RgVWkzFPSpTJ9bANMGCn4vd25OsE2LHbGR4wjhcdDKmoPFQ45P9KlkT8IqeLyYBzZUEMaBSVFHyk4vmqgsNrMCg54rmKbU+naXG+Y+DRSAFrjwDFCIwcfYqhJuYzRuggiAGpcjvYWpcK9P4NngD6VKBowiCmI+AxqvcbqplFMsm+KA9ZQCCYwmoIMUOxC8uvNLTjkI5TaHLOK4nUBpz6G5KeSoN/BfbuUjns1sIA6ApUSiFffvfknVHUmwP70ydS5NqiGmbCfqIwSjUJV/dhZHJiylCkW2YC2VcsbxPyqLWbUph/9b5eC3B/+olcHRmmICAm6UQSz0TBDjvOVTkijxV56Guz9C3ldc/6Bd0zI3aPiUtuhQFyTiw9atoDc/isAMoNxULM1+CrGbV1W4TDz1+UHXkT28h5tedOvR2KFsskyyPuQWXgNITVgU9ZiJ8C64FFZvGdyH5k0OLnbYJcB0GFSVo86kkfEgRHOH1ddy9q+fRZNDl9Pg/ttq7NkzwrImQQMicrBw6n/SSOZg0Yk6GQbOiKlWwhe7BjoGmLVSQi4WktM1xNJnxZDjmFTpVgOTQKIKCbJAn1RI6vIuXAn7++NCKtCx2hj04Hzarr9GxRZndlmfHrqz7auRL0aYwqpzUfToHIrXeTTtqbb3/7cEV9HG7pcHtnDEorHUnJIQimms6sL+Ge5dI3K0jQqAX2ply3KW6ZAn/IaY7puFWVfUlvne+qxgkzRjU0dRmnVo9qD3jtEMKeZjf0Deyq6TAClDQqsNyyaVm+btmfa+sFXjsPlejuUtEMAnqUJAD9EM1jCCBlA8U0Ouv50AvP83IXbK6DSzt7c1zrViSzWBTmOexnMRyuYq56D5UCAjIxtjJywBEbqW8v/v9xcjm7cOWU046fX3d+2og/N3kGDCOPWmzo/9uwOStMAXOIcA5cHwN8RLMXW36NvLwAN6ya/q92R7faBd6zNEzknMpOC7SfaIZO8B627gSfDxj96GSCvaAAlhrL3ElGvT6OTzq9bGCQkLJfOCaFwNXVhM+JiRrs9rOdN7TTpNv0BFFwM4ryl3HHlOdwFNcXxA6th/cx54XXEGwo2kRCbz6LLUX2fFJ5b2JXgyKPUSOpY5RyfNJACLj0twlM0PQOc5DoD4MnrGZec9sG0V1CwO4w6xO0ZNgGCb90nDREMUrskryGD6S61XuxcCfWifQ3fAxvzsodLkcA4t6XYUIOuYlsBt8+irvSMXn1OkxqfG0nMoMH2/AQgt8hEYmBkxiAFpT21Quixowxcl6G3wNlcAabdoYKpoYA4mXtp9tHtwA/RoFDp28HWJGubURpdXwTTEdr8AG0mRmlbldfn63VAFZOHWD1WCNdzvisHrQLL+6O4EX9Dltjh3N+tx+tWR1sR8DNLDojVtf/qFwYM2BZgPZvTKTTEd5AlBpiUrk3HThU0B9gstAvQQgF2UHGrTbM+x9iRojrkhAHk3F3Ddty+ePpDCN2Zdw7b6QJ1DtDCIax49Iwz6cg4QDic9IThEJxwxuB4h5AqjxJy0QxQvUlUw3KTH5wf2naUDg6zXa7JW3puC8dBtsDI67+7Y9CPxVJ80PF7OX6NyjPdgU4g1Heq0Tr8XEz+Urg7OYl5utIntsL3pLaAazZ/z87lhpn7uj+M8CzhQCe2MzCJxe/nmhZ5u2yje1VDPq7TK7NfT7j8P5GwfsAkNe43tJp6oOJL+0gzcZ5vqp5FZ/uRDf1Sac92SLPCWjgI1odzzWG2vqzaLApOrgEhJKl+b2KHiJ1aT5Ak0jiDuOFYVL/kjHYC2XJuFz7PFyNS1kmQShXsGMQlTjy9wYBzZhwcvvQDrztmMh0OLv2RjrsvQiEAU29Ds+xYcRzlLItSqlTgWHWJvDiaQK/jP2LQpoCkCIhZRoOMnsoc0nr0EcV+op8GdtBtrAEW0lzcgXw5aAGZ3+m1y7++OAxOAV3421mUPN+YdvAocIE+U1sD23fFLvLt73OCa5cxDhuNgrOzjh+g6uepnHn8cyf0Sq+F1H2cM3l7nlHF2SHWc1CW52WCqoj1OGbA8CwV9f9QD389+kg0NtUYOPOHhFeKfAcV/Fb/iRCJSaIdbv0ZG1soFWjiKKMfEWR4+skP4xg/LMXx05p81EtP9M9NJXBWN7pdbzTau0x7kTW2IcOBrhk6Cz1nXP8eKRYLSL/uTlFE4/gIIRbH1CipitwX4vCm3mlk9Qafai/lMqmFlieK47RMoP01W0KWpjGz1F7b5aIS9dwPgiNjCNHP9c6hHRhvLrfnWJqK1GMKHjIh0q10d6ytomkvznd12VSR2XlsEIDkjzGoO2+kOPFYv/Z6pJMjWjxqztGfmdv7XV18amv7CR87WK1GqBwlVHmAppyeu7gbKcxsCfY879nr3S9vEJ6/O2sVV/eWpmT2dHlO6TgRnr3pxZz6bVv7iesxUL0WFvdJr/0myEJzzX6gWmH1l65R/MY4eYxYCPbunGjq80PA0W7sfbqcGXX2MVt/02xnPWBuEuEYmPjT2Zr9RIRjUShZ09dQG87jnOtOMzqckHpjRrHbcwP13uih3Y1acB8pY1V6I236CButhOEYjRll6PQOlOGeP6RixyWo5llk5JnHYucx2W/a/L2BP15oheeTo5BHxNlzmEz7mxswshDzTneFYHSlwHxCskOjoGsdXSs/+cjaPpujh2M07bYmLbUj4yrk5j4uYeEbl4b6rTrgTeVpT0ubYyW99vOwuUTgOJs72A80sDj8RKPhspaj+482KCQsOJp25Mycuia3hCNHMBt0k6JeXoCr/w/cELTpiyQAAA=="
  },
  "ace_context_chunks.py": {
    "sha256": "d7e4b17dea0647d859bbeea5d7809099c5582f7ee2b3bde05aec385864aa862c",
    "gzip_base64": "H4sIAAAAAAAC/6U827LbRnLv5ysg7QOAc4YgD3VZm+dAVV5Fu3ElsVW2k0oVw0INgSGJFQjAuJyLFVXlH/KH+ZL0ZQYYgCDlLetBIgYzPd09fZ+GXr58+d37D7OfG1U626LNE5XM4iJv1FPjqKdSVelR5U3tyDxxjkrWbaUS4dTyWGZqJrN0n6vEqZu0iQ9pvg+urn5Sba1qpzkoJz6o+BO83sq6nidVe3Qei+qTqurA+ZjJfF5WxVY5idrJNmucpnDywknznapUHqvg6gelktrJ22P5DDvGKf2DKO7STAlntzuWai+cMmt5O/WUAiL53nn/7//0nVO1eQO4O7uicmYz9aTitgGgL1++vEqPZVE1jqz2paxqZZ4Psj5k6dY8/r0ucvP7KJvD1a4qjk4Jv2CSo198xBf6d31omzTrntot0Berur4yQ0SKI4Gmsptl6MHhesd7EK1BDbyVmdkoLqpKZbJBwtM8iUolP/WQVabipkrjaNdmWQSI7wkeMCNTcCTf/fLBCZ1Xy8VicfXTjz/+Ag+IuBdFuHUU+UGl6iJ7UJ4fAEvwwNe3m6urKzgbIEsu37z1kHB/deXAnyTdq7oJNb8CPcGnd49pc2DgtCAoSpV7brV1fcKoqZQ8MhT8g4ezzYr4E5y7kzaq8jJ53CZyxRMBLZl4t4vl62v8yxdb1/VXvH3Qlgmww6PlvHelmrbKNXrBQT3xL8CMKXmsYIsIz5WQEw8ya5WmCQfCHu+7RiFnZfUc4nOAZEV1u9ulT54bNMfS5S27aQEDR73xcIcggcOuPdpCwIEBT8OlkFlWPEa5zMO/yqxW/o37X/kJpEqVmYwVI6JRz+Rz0TZeUzQyE45W0PDbhQDRrPZpHr4FuZCJCoM338D7VtXhD0UOwqJyVe2f6UFTSijVoaeBCAYgcDVjku4cT+bPXgryUzcSdNF7ENuiyHwHjisvGmfwxkvzRuyyQjZ+NwH1JUhrkFRgivfg00E/4CHz7n4nAXrB7eI+1Ajdh28XCzO+uEe87hlHHORf97f4Wy+YLa/N6MLvZauSaa2c/8D9PlRVUXkuGhSzCHb8v//5X9ipNix8F97WzETE01toxvjI7krB20XtdgwacYHPBRjh9wRdo9rdh/SKaKKBS/j9XLRVrJxjWzcOGEaDoVMrQDoxu//J+R4o2CvgRZGozEnaSjZpkdeAdpy1gL4CW13KJEGD7PygHmCqeoqReDSTlfoVTqBBVsgyIJBH+ZQe22NIxwZHCchoNvmEtNaQJKzQXHnMGHrj3xF3Qg0BjgKmGSbhm/vbr9Nt6DFHA3g5mZIP4EYysJ+AtcyBG7VTMExHtklaaHbEgFIT3uJ+mtUaFzCKsA88eEtBhMUqzTw+KsbTnyM0X8ORpYzTJgXVWOPwDUxAoClKw3ohaJ/Z7YbBEhoo1PS6kvleeTTD3/TUI2Lh7cqGrHHjSeCRYGixuXPKIktjmmDe0G8CAz9B1BzUYL03jJhNRnranNXT5nfoadOrZa+Zi/vmnng2JzeCNDdIM2JBcPDHi7AGN6QSr1aNR28uKuJfUIzA1DnooetO4BkN4bR5CiKKzh5hUugBdKQg2Si/6N302Y8O4JYPyd5aHSUAzfeshjPk+BoOsZvQgCirJjTjNyzg3bLr/uzWKbyf1+3RG46tNn7PtqxA6fM6cFr4Be8+XjxYekjDY5r3S0c7axAsZLPUv9agewAgDVnx7pCuJgxf4ewgdku3EGdsDe8x1qvGFoE1cK7tbZYe08bYHYtdiOghFUhqVggeHHBBIDX2W1A4wlbopTx6sxzRUJRkycI1n0PDr4dCx4Teh4Mp9+Eh3dis0JBWtkSjwhHu+qX4pJ5DDjicZiW3tdfMDDFaJ59DF6K4B5C9JDKcczuYqI6jHUBYU4xL6jCDaNRjwSSG3NwKRHU+By9FirOeZuNmEuUe8BBrjx086C5SzQ9sJhb+TbBYLK8HdM2H/Ca7runEyITXR7mSVcQr3DFYdysztChJhMrb8wIFN5AlhHsJCrHfGzUzyhv5V+PZJNl6+NDmnzorSOotPCm2Ph6+gvhZVRjz/ZaWpCiCtIVUsD8GMHhVIwAwSeBCyBmaehI8dtNbNO09D4z/DIGVyYxWM5u6GZC+KJ0EsBMxS/wuaOnZQBQY0j67GPo9uatUuOgyIgLvrqR+hDnuaitgVtk25iUToMdoBvwlBmdm/XEpCogMSu5qAs0vg/j4s8vpW4S8dFdkoFxiTbSrJBhkd8WMcjVxEdiiyAQhKxM3Xg2xIIvRz9JRpYvxVD+KT8I1ihQZCXFX5hfsSQyEbejfLzr+xUQgIrc/SERkI0VdhfWOMwWK6mX2KJ/raJmEv1TgRZLmuVShS47v1bKP4OrqRUgeDYQM4WAaUypIe16ES+P78rJ3jjjHDyCA974eY/ISkERIZQvIuz7982+DmMUkKgBS04d+GhJkyBKiYudB/J/rXfoXIY4GoJKe2w+6oh8tIYkDhRWfv/in01xMwl2/I79/x5ECRLw8RbiYqcPUCT/yc1uW7JPpdCwg9YqyfOHoxTaZ/SyTh1EVIAL19vp3hl4btTBknFYakk5op+bxvv3J6MQYhyOqM1gxATN/+OYPEctkYS5dE1XW+endPLffL8D80PA5eJQP5jdl764/pK2Th45ItsOey9yISkDFgOzqAIifBj0Y65AFYcFM36sp5xAOZJdgbn5faklVA51fXgv7HLSUMel8zOH0adMMvTnvzdk3D1kFCUHDPMUapvU8Ghw/JSkwnQsXrPVUC4qKTzrP5tlVCuSAlcPCRairFno/mrBHP0cWMyLjGzL8uce22L3hyQHXAQJwmKrytEpx8SeIi/IZuWP4OgbZmx/e/uT9i3CA5iVT86Oe6NBKkFKMNRLn8YDlJD5frIZhhIcCZPIlyKxyQ5mrKUJJ0cnle5kXeRoDWLJYmHqgH0YIGBHPj+kTQi3y7Dlw/sboQ+iOdSvaCY1Q9aASDc5QA2dNOHFJDCGAn4oPulKIsRKCmoHfhHALS4ok3DMUboe0puZUtS+rBVWbe2uXy4Ag57O8qBvIefEnyT1yicbxr9QVIHvmuGFAVjzCYemkd4VJMaxd4vp4JeGfMj5G9e3bTPFa4qW/EVToJMFjHhPnQstn8cQ+tUPd6fJEo1H9WR9VI9E5hFRJAs+V1B6pQTfXJ4fH5SZ/kAOYtewCmN6Ipc0F6RqK/TB4vZCoGahOUsCR6dQxPtDhWUdM9QuUWytnIIKHWCXFY75VElILsd74NyOUwVxhdE7vhlBOkkyOApSJUD2KXyyqqDgY0hmsORqkGTMOxL/xVxgVZir3aIrPb2/M283YZ1AQ4UFUUP9a0b9HOFauQF5fL32ND6pKqAt2FuyJeptAKgTj3q81lc0axOooowdV1RieLkHkZNPWkJGgLqrEFZ2p4FMdmg6hz56Mta3op8I+kBIjIiyzgqMKbcWDjz99+PnDL8Iy+f3PU7iWdbORGRu9IAeui9PJk1SdbALCCPT9oK8OKoX3FBB4zTshC5z3VmFJF8lq7d1qXV7gCtSgcvY9hO4YxWOMxNHoHbnHNG/T5plvRNqaTOWvrcxwrM3htNJdqpJAa4BVczY2l86YHTY5djtOwAHtoU/44e2KDFJ2YUUXVLXW4/M+EpzmO2+3du0hyJxGPhUvfWLQsvASOG1PhgD14GZQJC25whTR5UX6oKKm0Pj2+1I2bF8zvAgNFpf83986tIzlMS4QwKkaDlt7F3AUw4gUNzGFdTCuEZJh5xWjy4Ae0zvSbcsqE3knppjR4ZsDDkvmE6y32aTVs7+GGbHMAmVzS5vxF+HkQVxiXu/mfy/vAFPeRecZkChNhnZdwOpra0H4kjsPOL8coz3IPS9i/ZG2JoCMtUlxnSTdgfbXHIhY4c6pFlmsnNC/0UttRLWwkHI/R2QMbG0UNNJF/Jg20BmDxVZH/hWFo/TAmmsLlFFmC8qEr0e4oZ7Z7XHXhQ282CSBGLpiUthd7tiv2adgZGDUHBwLMtiepE+8OzAjAfYtztCJjzyTWfoH7MnkNlwxAeB0AOtBHWUzLOKcrO2qnn3tBAB5A0hYfdnMpoD7X4GuadjLlMAGfz6dMirc9FSMXgzYPFIkOPsx9edxHsDRBoQBDCUC5Mkc28Vax3vKiylgz7EkOAehwAv4em5wn6PLdo5pTdHi0AjjPqa60+asVLVXFQVrXmTZY1vVtG217HY3+atZZ2+a7mwtjc5op97qsiO+sqoNuzhvsv4a3OiyG/QgAgwWXXMvLm/4XhwHrVtxBIP3cPEnD/8SPPCvP77/l+jDf/639fTDX3rlgHPYg17q7NxsbUY54DiZPPBlNoCBzSHbP3hL+XXt+VyL+PxlcKNv6iXaX+iK3maYbGAVm3IoI6hYJ92AhyXja/Df8eroM097SdNeblaLV8kXd1jGpnW6FtCl/31OZqVIsDVhpylanYSUsYTQO+kDKz1xDQs343hpBBudOS8/deHkvVeTzhgXkPPpnC8R7J7uMfBBvJPtg07mlxUEt55LbTmOiU41dLxKELusrQ/MqDsd3qoBlD85P6m4wFtkSZXNGnGVDTYsNI7cNfgipxC6qtoSC2aGYRz9BsMLjQJrzxhgdkmwLTsGbprr6z/Nt2CfFVtQGH4dXQ9MU5+mkTvVk2wfOnVU2uilLNIgyxjRW/qg39uKMPSZoe0zV5MlhMFpabzs47rr2aHf3m1hw09jke2mmavg0+20Z2DnUT43B0g5KIZFkwo6DOe4k2nmbNUOL69j2Ia6peDscN9nw/rTBKs9bsGiLk5ecKVJHxCoqjmcz7yCldTvbMWKh2/C21OF67hwGdgpCpyWmmNg96Frbx1QQf7vfIwwXf3p74X+oZBA6FazCdNzqr4dhuc0eGB62F6aNcFpSgV+ze5xsi22ME/DcFcPaj+cbD27J2pQ9VgurvMSNGN/u/CwlMKjPF3cqtkt1T+0Pz/WuuoxAWhcPqHnFmvRtERf2MAoTX/72u/hlthmOIB8LPKC6zsMTIKshbf+erX6ZhPIGkF5FigTAmNZBpf6929fU7ALCPMzUPJ61V2UHX5zV1TndrETskjSGLJ8d7UIvnS7z0L8m3fX1b847Lr1CKrQfyV4D4UlfgGB4KFI4GkH0eu6wwa7CTSIOXJAd4t4t6KfAsNS3yqbMZ0UZMWjOKT7g74Zf71YLObLNwu++rTGXr3xCZ6MYUNddsJuQoqCTGshvF0DwBUCRK+MYzcQdT3aWStCoTeUk8KKxeY+XPxu9m2xi5CFAtYSoA3SDIb2LpP7EB0X4l5yBZp8Nk6ixhdYsHkXIojr4Ns3pomnyHdpgu2jYSdscZaWRI3cb+aEo1iIW390JYqoEndgGhtcA+ldGLx5w3HOBC39PHNVSTeejwr4BkbPbkiRaJ/xWDOISEvs8FsIeCrT+ZIbV05F38YRSSngXBAKCLhAZDYIAAz/aNR0bip5jB7hSItHEJZdIypESmRkZ0TF/8QKHbd4TJMu3Nbmkl7M6MV8vrzDG3V6c0NDTFKIcNc0PMumjCLeXU+/YUnfhoSUBlGdBVGdB4HSwLe3kL4RbmJJArk9GZ240/wrNvyBd4dNapDI5wKCgG0BgZiu15rLzEcpHrfh4HCZZ3fbDFkjrx/lzfb6cctVNCkUkIZ20Bf4zxZTjh132JH585S8VvJGba/V1p8vjfxWVS+5bNEmzNg1rKD1W47NUUloR/8d2K8/98KqrzbBjYVg2wkPRNaH0Gjncwy0s5YsAn1r8xHGIaz7tU0V3Qlqa5Y4DZieOuXuGVBGyPDT35TmF6aAWcblGAyltm2aJa1uMQTRbUKPCMa+FG5LIcz8Gw+fvdsZ7uPrxsEKAx20MX3f22KUPuo2D0Ekic8uC6zVqEDP7JRdlNMIGBAl2x28A24oaV37uCSFowlbn9o02M9iawU+CJf260N7XuOuiBgLYlkpMBJYX8jUA4RkXGCjqQR8jpYOTk37TtyqbtwVnz7+9o1F4fb6SFaVfK4pPxXk8mqtr7prZpht3ZF0al/Arwa9GBur9Up7RF7qvwi7W4t6SmM+QAzzrJM807pHFaG8ce17WTRkv6kKbJY3VecDdRwZvFfLvq2PjZO+gswd6vgh9IakG/2nCMBo+++M17SVGDd6cHgx7vS4VAFhLGn3eY7+gLo/RlUPMpdia6JQqxtoI+wxwrev1BAn13K13ehrLDlJy2o7O28g0Q8Mm6rsnknr5C16e3/BGOsuRPN7c5cXRyzwafM9oKdvxVIS7LbpGgYjdRu80TW+Ub/QZv7KH/UCWq103lrvtumr0KdNRBtCMbzYsWf/IaPCXNDgZ4yw0I83/Ng17i0X1lUrZsWNBD5KjIPARjsHme34uxd93WS61tuma16l9nJYoSBxd1Syt5LjpkqPA3Uds4j4YzzFiEPrmLWGmiS7YbKlnQd/Z85qICI3uO8kf9CGxzfs5Tog/j27/IF2IYhB9yVXQQwa0x2pKJVcJ6N+CjjCvCGWOElax7JKLMWhSBEFyLNjGrYELJjm90ZYkU6sw5op3vh2H+XT6mn9arNmE7wZKKx2L5jz1xRuWhLwPbhe7FYQOoDQpw8eI9ZXhm4N59pSby0VSFLsZC1oXq4eSRyCsbIPhXElteYzoaO3M/swjC2QU6MnNmW7Gok58arfbLPezqYEZnLZ5MxpM0b8tE4WqxEDCk3b5gQR+uL+lf81Gqks9XVSpod1A8CrgUhf6sgauomsDKsyPJv98BDeWrsr9yN6dvtWWUc9knrLZvuqAPkyH965X4adx1kpYCNOipmPvtApMpJvGQpqjLpdLhbXFIBCFr/0qnKNKG7mmf5BoZd5IO03U0YBpc4Rmtp0KlBVNkyN3TQmK9TPHIJx9IEc5hAglDyEwRQPbM80uVLsRoSFSDJFavxYlUywCa6YUPr7DKyjzFvAsFIPqXoMPSJjfSak2zj3zuwVpbbb2kw9H9Zt/Hdvz3XpDv6gMaKPKu0QF/mN29AbiOMxcT9DAjcdaBoivMdKASlufJtewj0SP/MXhTtIcWsnKWhvhRUc7OBiUZs36XFbqcD5gC0NsxJb3Pqen32LpYdGQbx3kLWy5DawjDWFG6YDmrjWFxms5Aa0HGsZQLJu7tPOFa/MQg4WBGT2c1zGNxDwgyXxNrAzAV5+jesgC6Dd3RX9I1ysGmPniJIJ0Hfk+7gVTcWTpGcqeLgr/Ptsl7VjFFbzMEnlHvvN0rimVrg7/HZVtk0B+pXGTpzJ9EifKx2KCs41jttKxs/CYfYi42pqeYuf6V5w2EYSuKPIHzvd/sid2PCyiy74sOVm8tJr6mbIm75NmriYZosLseaduWqw486vXgwBoae3LhO3RJO3MDruABjTDSeXL5OmMh19wWlfy/T9EN09/KCkC+8ni7nMGKMVVmsg3cD6HbvMDALE7+yUyhwEC4ZKXJO/6VbYrtY+0c+sl+ip/Ll0fdd1bDppY+4GJPDxAIrPu7jW0Qp9xXI+JT1tzNWXqTv+RNa0Qe3cz50gfuG+NN6BnEXdbrlv/+P7f4uWr91pGO4xfaKmaV4fvLn2gj9f28zlRpMb3YJ3ATSKJ+JDX/t1iIluB1tK/2AP6pABTLvftZhm6fZYvoK4VOGSLY3dfrv8NL0WproTDahEDcbXmOfRia21YbTI4PqITjnw7XpcQtmMUrLTM6Rw/DMtHqiTdaA6ntPlxFcMctx4yW9v9NvNpZPqDp+VmcmrwRjZ0taXEHpFMnWEofERRvPGnxDpz4e8iYT9wvWP9RWcyvfNgTzZN8EQzKAmYQMew5joHhxSjZIwYc10e+HAm33mpfrLGmH3eUSGB+A3rb6QkTdEh0kRB36BqSO5vth2SszU3VoPwyxkNgm7KYiJc1e2B4JTPN8T9cX/w7qJX75Qgzh9/2U1jJ8ywp01/I5R/4dV97wTitj+Rmxdzym3vpc2XcK6KbjzCNPNuV3SItQTc5HKcKGlgxOxT9dbFk6Ya0uATM/wWdvun+2G5WiR5ZUpG5R5eUiHQ/hVrinKlaH5/zuC76o90fgRnyovAV9dpfSJZxhFSRFHkX8HchFCtJkkEUoITaxxZhOCzB4hK0hweyuStnq5YQWt5GWey+2F1AtK41Lv77mzGXscV5Dhwuv3EdTpVQW26J1b0jeGj5eZ/wDBKBODoBBb6P9WJfx2cWZPLi9dXPv2wt6Yu11cHLz55szO2PVukXthE/sLNgil0xhLXcPv1Dbdhvrjo87+o/SRO3fpAxJYgcGb0JoycOjjI8alIDMn+NDJXz4oE4kChBdGK8GsnoDCTgUblKEC/3+Wqa11WwE4Vfo8AkGjqQD2KPdkZy6CAs0TG3d9hbw5/q8AHWvfLsS3C3G7XGzGYnsKh6381MnrixqYXIPOEUtxZe11tw34FGi9C/mDBtf+ar9GMOYTNZqsezroN2iL0BD4+1Qthjw4LJvymH3zoZfilxeWpaTB8fdpKpvAlflqNbxdyMNobZ9b9X1qfX8bwYs+00xzNJyR3P2eljYuvfS9OWd7cmjiSUsOjdodOZr3X+vs7hpuCG/qurncJikuYqQ7dPik6WKv674ZMKZrrqFR/XD+rFDfh/dE1rKVJtVqPu2JmTg4KrlFgleJ6Pwp00QN3ErgR7O4L8/6r4I0YPN/BVHTC6AckReNIiAnitABRhF+XY2O8Or/AW5E1kW9SwAA"
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_chunks_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
modules = {}
for name in ("electric_full_song", "drum_probe", "ace_context_chunks"):
    spec = importlib.util.spec_from_file_location(name, TOOLS / (name + ".py"))
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    sys.modules[name] = module; modules[name] = module
context = modules["ace_context_chunks"]
ROOT = modules["electric_full_song"].discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = modules["electric_full_song"].discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ada. Pakai VM aktif atau sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    modules["electric_full_song"].model_python(ROOT)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli seluruh lagu"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
DRUM_CHUNK_OUT = globals().get("DRUM_CHUNK_OUT")
if DRUM_CHUNK_OUT is None:
    DRUM_CHUNK_OUT = ROOT / "runs" / ("drum_chunks90_" + uuid.uuid4().hex[:10])
    drum_plan = context.prepare(SOURCE_PATH, DRUM_CHUNK_OUT, context=90, margin=6, fade=.58, instrument="drums")
else:
    DRUM_CHUNK_OUT = Path(DRUM_CHUNK_OUT)
    _, drum_plan, _ = context.load_plan(DRUM_CHUNK_OUT / "plan.json")
    assert context.instrument_of(drum_plan) == "drums" and drum_plan["context_cap_seconds"] == 90
    assert drum_plan["original_sha256"] == context.sha256(SOURCE_PATH)
DRUM_CHUNK_PLAN = DRUM_CHUNK_OUT / "plan.json"
print("Runtime dipakai kembali:", ROOT)
print("Plan drum:", DRUM_CHUNK_PLAN)
for c in drum_plan["chunks"]:
    print(c["index"] + 1, "model", c["model_duration"], "detik; core", round(c["core_start"]/32000, 3), "sampai", round(c["core_end"]/32000, 3))


## Generasi tiga konteks dan penyambungan

Jalankan sel ini lagi jika satu chunk gagal dan VM masih hidup: chunk selesai dipakai kembali setelah verifikasi. Tiap chunk memuat model; margin/reload menambah waktu. Tidak ada fallback otomatis ke durasi/seed lain.


In [ ]:
from IPython.display import Audio, display
if DRY_RUN:
    print("DRY RUN: plan saja; tidak menjalankan model atau stitching")
else:
    progress = context.run_chunks(ROOT, DRUM_CHUNK_PLAN)
    finished = DRUM_CHUNK_OUT / "stitched/seams.json"
    if finished.is_file():
        drum_seams = json.loads(finished.read_text())
        assert context.sha256(DRUM_CHUNK_OUT / "stitched/drums_full.wav") == drum_seams["stem_sha256"]
    else:
        drum_seams = context.stitch_run(DRUM_CHUNK_PLAN)
    print("Sambungan:", [(s["index"], round(s["center_seconds"], 3)) for s in drum_seams["seams"]])
    print("Seluruh sambungan perlu didengar; flag numerik bukan penilaian groove.")
    for name in ("drums_full.mp3", "mix_full.mp3"):
        display(Audio(filename=str(DRUM_CHUNK_OUT / "stitched" / name), normalize=False))


## Simpan hasil dan kirim feedback

Paket feedback berisi mix/drum penuh MP3, cuplikan tiap chunk sebelum stitching, cuplikan sambungan, serta report aktual tiap generasi. Paket result juga menyimpan WAV drum tiap chunk dan WAV hasil akhir. Keduanya tidak menyertakan bobot model atau piano asli.


In [ ]:
stitched = DRUM_CHUNK_OUT / "stitched"
if (stitched / "seams.json").is_file():
    for filename, full_result in [("drum_chunked_feedback.zip", False), ("drum_chunked_result.zip", True)]:
        bundle = DRUM_CHUNK_OUT / filename
        with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(DRUM_CHUNK_PLAN, "plan.json")
            archive.write(DRUM_CHUNK_OUT / "progress.json", "progress.json")
            for path in sorted(stitched.rglob("*")):
                if path.is_file() and (path.suffix in {".json", ".mp3"} or path.name.startswith("seam_") or full_result and path.suffix == ".wav"):
                    archive.write(path, "stitched/" + str(path.relative_to(stitched)))
            if full_result:
                saved = json.loads((DRUM_CHUNK_OUT / "progress.json").read_text())
                for c in drum_plan["chunks"]:
                    folder = (DRUM_CHUNK_OUT / saved[str(c["index"])]).resolve()
                    assert folder.is_relative_to(DRUM_CHUNK_OUT / f'chunk_{c["index"]:03d}')
                    stem = context.verify_chunk(folder, drum_plan, c)
                    archive.write(stem, f'chunks/chunk_{c["index"]:03d}_drums.wav')
        print(filename, round(bundle.stat().st_size/1024**2, 2), "MiB")
        files.download(str(bundle))
else:
    print("Belum ada hasil lengkap. Tidak ada audio hasil yang dibuat oleh dry-run.")
